# 01. Intake, panel, audit and features
Run setup first. Use the same Drive project folder. Run one stage cell at a time and share its output for review. Never use Run all across stages.


## 1. Mount Drive
Authorize the account containing the existing project data. No temporary-storage fallback is used.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, hashlib, csv
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/FoodPriceWatch')
assert Path('/content/drive/MyDrive').is_dir(), 'STOP: Drive is not mounted'
print('Project root:', ROOT)
print('Existing project folder:', ROOT.exists())


In [ ]:
assert (ROOT / 'src/fpw/runner.py').is_file(), 'Run notebook 00 first'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
sys.path.insert(0, str(ROOT / 'src'))
def git(*args):
    return subprocess.check_output(['git', *args], cwd=ROOT, text=True).strip()


## 3. Install runtime packages


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-colab.txt')], check=True)
os.environ.setdefault('FPW_THREADS', str(os.cpu_count() or 2))
print('Package install: PASS; threads:', os.environ['FPW_THREADS'])


## 4. Startup verification
A failed check stops execution. Missing or changed finished outputs are reset to pending by the project checker; paste failures back before continuing.


In [ ]:
for folder in ['outputs', 'data']:
    for tmp in (ROOT / folder).rglob('*.tmp'):
        assert tmp.resolve().is_relative_to(ROOT.resolve())
        tmp.unlink()
        print('Removed interrupted-write temporary file:', tmp.relative_to(ROOT))
subprocess.run([sys.executable, '-m', 'fpw.check'], cwd=ROOT, check=True)
def run_unit(sid):
    subprocess.run([sys.executable, '-m', 'fpw.runner', sid], cwd=ROOT, check=True)
    ledger = json.loads((ROOT / 'state/stages.json').read_text())
    stage = next(s for s in ledger['stages'] if s['id'] == sid)
    assert stage['status'] == 'done', 'STOP: stage was not finalized'
    summary = []
    for name, expected in stage['output_sha256'].items():
        path = ROOT / name
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        assert actual == expected, 'STOP: output hash mismatch: ' + name
        rows = None
        if path.suffix == '.csv':
            with path.open(newline='') as f:
                rows = sum(1 for _ in csv.DictReader(f))
        elif path.suffix == '.parquet':
            import pyarrow.parquet as pq
            rows = pq.read_metadata(path).num_rows
        summary.append(dict(file=name, rows=rows, sha256=actual))
    assert summary, 'STOP: no finalized output hashes'
    print(json.dumps(dict(unit=sid, outputs=summary, check='PASS', git_commit=git('rev-parse', 'HEAD')), indent=2))


## 5. S00 — refresh intake
Before running, place the extended CSV `tcb_dhaka_daily_prices_2021_2026_v2.csv` in `inbox/project/`.
Existing original project files must be in `data/raw/project/` or `inbox/project/`.
This refreshes the intake manifest and its finalized hashes. Paste the output back and stop.


In [ ]:
subprocess.run([sys.executable, 'src/stages/s00_intake.py'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'src/stages/finalize.py', 'S00'], cwd=ROOT, check=True)
run_unit('S00')


## 6. S22 — extended panel
Run after S00 output is reviewed. The earlier S01 panel must already exist on Drive for the equality checks. Audit tables from the work-PC audit must be available. Missing prerequisites require agent review; do not rerun the old benchmark.


In [ ]:
run_unit('S22')


## 7. S23 — kept features
Run after S22 output is reviewed. Requires the earlier driver artifacts and feature table for comparison. Paste the output back and stop.


In [ ]:
run_unit('S23')
